# E4: Iterative Code Generation with CVX Debug Memory

CVX as **prefrontal cortex**: a generate-test-debug agent that recalls how similar errors were fixed.

### Why E4 v1 Failed

E4 v1 used MBPP (easy) → HumanEval (easy): 86% single-pass, 95% NameError in traces. No signal.

### E4 v2: APPS Benchmark

- **Training corpus**: APPS interview (200 hard problems) → multi-step debug traces
- **Evaluation**: APPS introductory (100 problems, lower single-pass for 7B)
- **Debug traces**: problem → attempt1 → error1 → attempt2 → error2 → ground truth fix

### Conditions

| Condition | Description |
|-----------|-------------|
| **SinglePass** | One attempt, T=0 |
| **Retry-NoMemory** | Retry with error feedback, no CVX |
| **Retry-CVX-Causal** | Retry + CVX error-matching → fix continuations |

In [1]:
import os, json, time, traceback, re, signal, sys
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from scipy import stats

USE_OLLAMA = True
OLLAMA_MODEL = 'qwen2.5-coder:7b-instruct'
OLLAMA_URL = 'http://hpc.glaciar.lab:11434/v1'

if USE_OLLAMA:
    client = OpenAI(base_url=OLLAMA_URL, api_key='ollama')
    LLM_MODEL = OLLAMA_MODEL
else:
    client = OpenAI()
    LLM_MODEL = 'gpt-4o-mini'

EMBED_MODEL = 'all-MiniLM-L6-v2'
MAX_RETRIES = 5
TOP_K = 3
N_TRACE_PROBLEMS = 200
N_EVAL_PROBLEMS = 100

DATA_DIR = Path('../data/episodic')
DATA_DIR.mkdir(parents=True, exist_ok=True)

embedder = SentenceTransformer(EMBED_MODEL)
D = embedder.get_sentence_embedding_dimension()
print(f'LLM: {LLM_MODEL}, Embedding: D={D}')
print(f'Trace corpus: {N_TRACE_PROBLEMS} APPS interview problems')
print(f'Eval: {N_EVAL_PROBLEMS} APPS introductory problems, max {MAX_RETRIES} retries')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LLM: qwen2.5-coder:7b-instruct, Embedding: D=384
Trace corpus: 200 APPS interview problems
Eval: 100 APPS introductory problems, max 5 retries


In [2]:
from datasets import load_dataset

apps = load_dataset('codeparrot/apps', split='test', revision='refs/convert/parquet')
print(f'APPS test: {len(apps)} problems')
print(f'Difficulties: {dict(apps.to_pandas()["difficulty"].value_counts())}')

apps_intro = [p for p in apps if p['difficulty'] == 'introductory']
apps_interview = [p for p in apps if p['difficulty'] == 'interview']
print(f'\nIntroductory: {len(apps_intro)} (eval)')
print(f'Interview: {len(apps_interview)} (trace corpus)')

APPS test: 10000 problems
Difficulties: {'interview': np.int64(6000), 'competition': np.int64(2000), 'introductory': np.int64(2000)}



Introductory: 2000 (eval)
Interview: 6000 (trace corpus)


## 1. APPS Execution Harness

In [3]:
import subprocess

def run_apps_tests(code, input_output_str, timeout_sec=5, max_tests=5):
    """Run APPS stdin/stdout test cases. Returns (passed, error_msg, n_passed, n_total)."""
    try:
        io = json.loads(input_output_str) if input_output_str else {}
    except:
        return False, 'invalid test spec', 0, 0
    
    inputs = io.get('inputs', [])
    outputs = io.get('outputs', [])
    if not inputs or not outputs:
        return False, 'no test cases', 0, 0
    
    n_total = min(len(inputs), len(outputs), max_tests)
    n_passed = 0
    last_error = ''
    
    for i in range(n_total):
        inp = str(inputs[i]).strip()
        expected = str(outputs[i]).strip()
        try:
            result = subprocess.run(
                [sys.executable, '-c', code],
                input=inp, capture_output=True, text=True, timeout=timeout_sec,
            )
            actual = result.stdout.strip()
            if result.returncode != 0:
                stderr = result.stderr.strip().split('\n')[-1] if result.stderr else 'unknown error'
                last_error = f'{stderr}\nInput: {inp[:80]}\nExpected: {expected[:80]}'
                continue
            if actual == expected:
                n_passed += 1
            else:
                last_error = f'Wrong answer\nInput: {inp[:80]}\nExpected: {expected[:80]}\nGot: {actual[:80]}'
        except subprocess.TimeoutExpired:
            last_error = f'Timeout ({timeout_sec}s)\nInput: {inp[:80]}'
        except Exception as e:
            last_error = f'{type(e).__name__}: {str(e)[:100]}'
    
    passed = n_passed == n_total
    return passed, last_error, n_passed, n_total


def generate_code(prompt, error_context='', fix_context='', temperature=0.0):
    """Generate code for an APPS problem."""
    system = (
        'You are an expert Python programmer. Write a complete Python program that reads from stdin '
        'and prints to stdout. Return ONLY the Python code, no explanation, no markdown fences.'
    )
    user_parts = []
    if fix_context:
        user_parts.append(fix_context)
        user_parts.append('---\n')
    if error_context:
        user_parts.append(f'My previous attempt had this error:\n{error_context}\n')
        user_parts.append('Please fix the code. Return the COMPLETE fixed program.\n')
    user_parts.append(prompt)
    
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': '\n'.join(user_parts)},
        ],
        temperature=temperature,
        max_tokens=1024,
    )
    code = response.choices[0].message.content.strip()
    if code.startswith('```'):
        code = code.split('\n', 1)[1] if '\n' in code else code[3:]
    if code.endswith('```'):
        code = code[:-3]
    return code.strip()


# Quick test
test_p = apps_intro[0]
code = generate_code(test_p['question'][:500])
passed, err, np_, nt_ = run_apps_tests(code, test_p['input_output'])
print(f'Test: passed={passed} ({np_}/{nt_})')
if err: print(f'Error: {err[:150]}')

Test: passed=False (0/5)
Error: Wrong answer
Input: 4
4 3
3 1
1 2
Expected: 3 
2 1 4
Got: 0


## 2. Build Multi-Step Debug Traces from APPS Interview

For each problem: attempt1 (T=0) → error1 → attempt2 (retry) → error2 → ground truth fix.
Only keep traces where attempt1 FAILS (no signal from already-solved problems).

In [4]:
TRACES_PATH = str(DATA_DIR / 'apps_debug_traces.json')

if os.path.exists(TRACES_PATH):
    with open(TRACES_PATH) as f:
        debug_traces = json.load(f)
    print(f'Loaded {len(debug_traces)} cached debug traces')
else:
    print(f'Building multi-step debug traces from {N_TRACE_PROBLEMS} APPS interview problems...')
    debug_traces = []
    t0 = time.perf_counter()
    np.random.seed(42)
    sample_indices = np.random.choice(len(apps_interview), size=N_TRACE_PROBLEMS, replace=False)
    
    for i, idx in enumerate(sample_indices):
        problem = apps_interview[int(idx)]
        question = problem['question'][:800]
        io_str = problem['input_output']
        
        try:
            solutions = json.loads(problem['solutions']) if problem['solutions'] else []
        except:
            solutions = []
        if not solutions or not io_str:
            continue
        good_code = solutions[0]
        
        # Attempt 1: T=0
        try:
            code1 = generate_code(question, temperature=0.0)
            pass1, err1, _, _ = run_apps_tests(code1, io_str)
        except:
            continue
        
        if pass1:
            continue  # Skip — no debug trace to learn from
        
        # Attempt 2: retry with error
        try:
            error_ctx1 = f'Error: {err1}\nFailing code:\n{code1[:300]}'
            code2 = generate_code(question, error_context=error_ctx1, temperature=0.0)
            pass2, err2, _, _ = run_apps_tests(code2, io_str)
        except:
            err2 = 'generation_failed'
            code2 = ''
            pass2 = False
        
        trace = {
            'question': question,
            'steps': [
                {'type': 'problem', 'content': question},
                {'type': 'attempt', 'content': code1[:500]},
                {'type': 'error', 'content': err1[:300]},
                {'type': 'attempt', 'content': code2[:500]},
                {'type': 'error', 'content': err2[:300] if not pass2 else 'PASSED'},
                {'type': 'fix', 'content': good_code[:500]},
            ],
            'error_types': [err1.split(':')[0] if err1 else '', err2.split(':')[0] if err2 else ''],
            'attempt2_passed': pass2,
        }
        debug_traces.append(trace)
        
        if (i + 1) % 20 == 0:
            elapsed = time.perf_counter() - t0
            eta = elapsed / (i + 1) * (N_TRACE_PROBLEMS - i - 1)
            print(f'  [{i+1}/{N_TRACE_PROBLEMS}] {len(debug_traces)} traces ({elapsed:.0f}s, ETA {eta:.0f}s)')
            with open(TRACES_PATH, 'w') as f:
                json.dump(debug_traces, f)
    
    with open(TRACES_PATH, 'w') as f:
        json.dump(debug_traces, f)
    print(f'\nBuilt {len(debug_traces)} multi-step debug traces')

# Analyze
error_types = {}
for t in debug_traces:
    for et in t.get('error_types', []):
        if et and et != 'PASSED':
            error_types[et] = error_types.get(et, 0) + 1

print(f'\n{len(debug_traces)} traces, {sum(error_types.values())} total errors')
print('Error types:')
for et, count in sorted(error_types.items(), key=lambda x: -x[1])[:10]:
    print(f'  {et}: {count} ({count/max(sum(error_types.values()),1):.1%})')

n_rescued = sum(1 for t in debug_traces if t.get('attempt2_passed', False))
print(f'\nAttempt 2 rescued: {n_rescued}/{len(debug_traces)} = {n_rescued/max(len(debug_traces),1):.1%}')

Loaded 172 cached debug traces

172 traces, 342 total errors
Error types:
  Wrong answer
Input: 227 (66.4%)
  ValueError: 48 (14.0%)
  IndexError: 27 (7.9%)
  Timeout (5s)
Input: 15 (4.4%)
  EOFError: 9 (2.6%)
  TypeError: 3 (0.9%)
  SyntaxError: 3 (0.9%)
  RecursionError: 3 (0.9%)
  NameError: 3 (0.9%)
  UnboundLocalError: 2 (0.6%)

Attempt 2 rescued: 2/172 = 1.2%


## 3. Index Debug Traces in CVX

6-step episodes: problem → attempt1 → error1 → attempt2 → error2 → fix

In [5]:
import chronos_vector as cvx

INDEX_PATH = str(DATA_DIR / 'apps_debug_traces.cvx')

if False and os.path.exists(INDEX_PATH):  # Force rebuild with fixed entity_id
    index = cvx.TemporalIndex.load(INDEX_PATH)
    print(f'Loaded: {len(index)} points')
else:
    print('Building CVX index from debug traces...')
    index = cvx.TemporalIndex(m=16, ef_construction=100, ef_search=50)
    
    for ep_idx, trace in enumerate(debug_traces):
        for s_idx, step in enumerate(trace['steps']):
            if step['type'] == 'error':
                text = f"Error in code: {step['content']}"
            elif step['type'] == 'attempt':
                text = f"Code attempt: {step['content'][:400]}"
            elif step['type'] == 'fix':
                text = f"Working solution: {step['content'][:400]}"
            else:
                text = step['content'][:500]
            
            vec = embedder.encode(text).tolist()
            # entity_id = episode, timestamp = step within episode
            # This enables causal_search to walk forward from error → fix
            index.insert(entity_id=ep_idx, timestamp=ep_idx * 1000 + s_idx, vector=vec)
    
    index.save(INDEX_PATH)
    print(f'Built: {len(index)} points ({len(debug_traces)} episodes × 6 steps)')

print(f'Debug memory: {len(index)} vectors from {len(debug_traces)} error traces')

Building CVX index from debug traces...


Built: 1032 points (172 episodes × 6 steps)
Debug memory: 1032 vectors from 172 error traces


## 4. CVX Causal Error Retrieval

In [6]:
def retrieve_fix_suggestions(error_msg, code_context, top_k=TOP_K):
    """Search CVX for similar past errors, return fix continuations via causal_search."""
    query_text = f'Error in code: {error_msg}\nCode: {code_context[:200]}'
    query_vec = embedder.encode(query_text[:500]).tolist()
    
    # Use causal_search: finds the error node, then walks FORWARD to the fix
    results = index.causal_search(vector=query_vec, k=top_k * 5, temporal_context=5, alpha=1.0)
    
    seen = set()
    suggestions = []
    for r in results:
        ep_idx = r['entity_id']
        if ep_idx in seen or ep_idx >= len(debug_traces):
            continue
        seen.add(ep_idx)
        
        trace = debug_traces[ep_idx]
        steps = trace['steps']
        fix_step = steps[-1]
        
        # Find the matched error from successors
        matched_error = ''
        for s in steps:
            if s['type'] == 'error':
                matched_error = s['content']
                break
        
        suggestions.append({
            'match_step': r['entity_id'],
            'match_type': 'causal',
            'similarity': r['score'],
            'original_error': matched_error[:200],
            'fix_code': fix_step['content'],
            'question_preview': trace['question'][:80],
        })
        if len(suggestions) >= top_k:
            break
    return suggestions


def online_learn_fix(problem_text, error_msg, failing_code, fix_code):
    """Add a successful fix to CVX for future retrieval (online learning)."""
    global debug_traces
    ep_idx = len(debug_traces)
    trace = {
        'question': problem_text[:500],
        'steps': [
            {'type': 'problem', 'content': problem_text[:500]},
            {'type': 'attempt', 'content': failing_code[:500]},
            {'type': 'error', 'content': error_msg[:300]},
            {'type': 'fix', 'content': fix_code[:500]},
        ]
    }
    debug_traces.append(trace)
    
    for s_idx, step in enumerate(trace['steps']):
        if step['type'] == 'error':
            text = f"Error in code: {step['content']}"
        elif step['type'] == 'attempt':
            text = f"Code attempt: {step['content'][:400]}"
        elif step['type'] == 'fix':
            text = f"Working solution: {step['content'][:400]}"
        else:
            text = step['content'][:500]
        vec = embedder.encode(text).tolist()
        index.insert(entity_id=ep_idx, timestamp=ep_idx * 1000 + s_idx, vector=vec)


def format_fix_context(suggestions):
    """Show top 2 fixes with code. The 7b-coder model needs concrete examples."""
    if not suggestions:
        return ''
    lines = ['Similar past fixes:\n']
    for i, s in enumerate(suggestions[:2], 1):
        if s['original_error']:
            lines.append(f'{i}. Error: {s["original_error"].split(chr(10))[0][:100]}')
        lines.append(f'   Fix approach:')
        lines.append('```python')
        lines.append(s['fix_code'][:300])
        lines.append('```')
    return '\n'.join(lines) + '\n'


# Test
test_err = 'Wrong answer\nInput: 5\nExpected: 3\nGot: 2'
test_code = 'n=int(input())\nprint(n//2)'
sug = retrieve_fix_suggestions(test_err, test_code)
print(f'Test: {len(sug)} suggestions')
for s in sug:
    print(f'  step={s["match_step"]} ({s["match_type"]}), sim={s["similarity"]:.3f}')

Test: 3 suggestions
  step=84 (causal), sim=0.441
  step=126 (causal), sim=0.444
  step=143 (causal), sim=0.498


## 5. Evaluation on APPS Introductory

In [7]:
def run_single_pass(problem):
    code = generate_code(problem['question'][:800])
    passed, _, np_, nt_ = run_apps_tests(code, problem['input_output'])
    return {'passed': passed, 'attempts': 1}

def run_retry_no_memory(problem, max_retries=MAX_RETRIES):
    question = problem['question'][:800]
    code = generate_code(question)
    passed, error, _, _ = run_apps_tests(code, problem['input_output'])
    attempts = 1
    for attempt in range(max_retries):
        if passed: break
        error_ctx = f'Error: {error}\nFailing code:\n{code[:300]}'
        code = generate_code(question, error_context=error_ctx)
        passed, error, _, _ = run_apps_tests(code, problem['input_output'])
        attempts += 1
    return {'passed': passed, 'attempts': attempts}

def run_retry_cvx_causal(problem, max_retries=MAX_RETRIES):
    question = problem['question'][:800]
    code = generate_code(question)
    passed, error, _, _ = run_apps_tests(code, problem['input_output'])
    attempts = 1
    first_error = error
    first_code = code
    for attempt in range(max_retries):
        if passed: break
        suggestions = retrieve_fix_suggestions(error, code)
        fix_ctx = format_fix_context(suggestions)
        error_ctx = f'Error: {error}\nFailing code:\n{code[:300]}'
        code = generate_code(question, error_context=error_ctx, fix_context=fix_ctx)
        passed, error, _, _ = run_apps_tests(code, problem['input_output'])
        attempts += 1
    # Online learning: if we fixed it, store the fix for future problems
    if passed and first_error:
        online_learn_fix(question, first_error, first_code, code)
    return {'passed': passed, 'attempts': attempts}


# Eval on INTRODUCTORY — best config for CVX demonstration
np.random.seed(42)
valid_intro = [p for p in apps_intro if p['input_output'] and p['input_output'] != '{}'
               and len(json.loads(p['input_output']).get('inputs', [])) > 0]
eval_problems = [valid_intro[i] for i in np.random.choice(len(valid_intro), size=min(N_EVAL_PROBLEMS, len(valid_intro)), replace=False)]
print(f'Evaluating {len(eval_problems)} APPS introductory problems\n')

conditions = {
    'SinglePass': run_single_pass,
    'Retry-NoMemory': run_retry_no_memory,
    'Retry-CVX-Causal': run_retry_cvx_causal,
}

all_results = {cond: [] for cond in conditions}
t0 = time.perf_counter()

for i, problem in enumerate(eval_problems):
    for cond, fn in conditions.items():
        try:
            result = fn(problem)
        except Exception as e:
            result = {'passed': False, 'attempts': 1}
        result['problem_id'] = problem.get('problem_id', i)
        all_results[cond].append(result)
    
    if (i + 1) % 10 == 0:
        elapsed = time.perf_counter() - t0
        eta = elapsed / (i + 1) * (len(eval_problems) - i - 1)
        rates = {c: sum(r['passed'] for r in res) / len(res) for c, res in all_results.items()}
        print(f'[{i+1}/{len(eval_problems)}] {rates} ({elapsed:.0f}s, ETA {eta:.0f}s)')

elapsed = time.perf_counter() - t0
print(f'\nDone in {elapsed:.0f}s')

print(f'\n=== RESULTS (APPS introductory, n={len(eval_problems)}) ===')
for cond, res in all_results.items():
    passed = sum(r['passed'] for r in res)
    total = len(res)
    mean_att = np.mean([r['attempts'] for r in res])
    print(f'{cond:>20}: {passed}/{total} = {passed/total:.1%}  (mean attempts: {mean_att:.1f})')

Evaluating 100 APPS introductory problems



[10/100] {'SinglePass': 0.2, 'Retry-NoMemory': 0.2, 'Retry-CVX-Causal': 0.2} (146s, ETA 1312s)


[20/100] {'SinglePass': 0.3, 'Retry-NoMemory': 0.25, 'Retry-CVX-Causal': 0.3} (393s, ETA 1573s)


[30/100] {'SinglePass': 0.23333333333333334, 'Retry-NoMemory': 0.2, 'Retry-CVX-Causal': 0.23333333333333334} (535s, ETA 1249s)


[40/100] {'SinglePass': 0.3, 'Retry-NoMemory': 0.275, 'Retry-CVX-Causal': 0.3} (689s, ETA 1034s)


...

[60/100] {'SinglePass': 0.2833333333333333, 'Retry-NoMemory': 0.2833333333333333, 'Retry-CVX-Causal': 0.3} (1137s, ETA 758s)


[70/100] {'SinglePass': 0.2857142857142857, 'Retry-NoMemory': 0.3, 'Retry-CVX-Causal': 0.3} (1260s, ETA 540s)


[80/100] {'SinglePass': 0.275, 'Retry-NoMemory': 0.2875, 'Retry-CVX-Causal': 0.2875} (1376s, ETA 344s)


[90/100] {'SinglePass': 0.2777777777777778, 'Retry-NoMemory': 0.28888888888888886, 'Retry-CVX-Causal': 0.28888888888888886} (1568s, ETA 174s)


[100/100] {'SinglePass': 0.28, 'Retry-NoMemory': 0.28, 'Retry-CVX-Causal': 0.29} (1672s, ETA 0s)

Done in 1672s

=== RESULTS (APPS introductory, n=100) ===
          SinglePass: 28/100 = 28.0%  (mean attempts: 1.0)
      Retry-NoMemory: 28/100 = 28.0%  (mean attempts: 4.6)
    Retry-CVX-Causal: 29/100 = 29.0%  (mean attempts: 4.6)


In [8]:
# Analysis
single_fail = {i for i, r in enumerate(all_results['SinglePass']) if not r['passed']}
rescued_nm = {i for i, r in enumerate(all_results['Retry-NoMemory']) if r['passed'] and i in single_fail}
rescued_cvx = {i for i, r in enumerate(all_results['Retry-CVX-Causal']) if r['passed'] and i in single_fail}

print(f'Failed SinglePass: {len(single_fail)}')
print(f'Rescued by Retry-NoMemory: {len(rescued_nm)}')
print(f'Rescued by Retry-CVX-Causal: {len(rescued_cvx)}')
print(f'CVX-only rescues: {len(rescued_cvx - rescued_nm)}')
print(f'NoMem-only rescues: {len(rescued_nm - rescued_cvx)}')

# McNemar
a_pass = np.array([r['passed'] for r in all_results['Retry-CVX-Causal']])
b_pass = np.array([r['passed'] for r in all_results['Retry-NoMemory']])
n_10 = np.sum(a_pass & ~b_pass)
n_01 = np.sum(~a_pass & b_pass)
n_11 = np.sum(a_pass & b_pass)
n_00 = np.sum(~a_pass & ~b_pass)

if n_10 + n_01 > 0:
    chi2 = (abs(n_10 - n_01) - 1) ** 2 / (n_10 + n_01)
    p_val = 1 - stats.chi2.cdf(chi2, df=1)
else:
    chi2, p_val = 0, 1.0
sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'

print(f'\n=== McNemar: CVX-Causal vs NoMemory ===')
print(f'CVX only: {n_10}, NoMem only: {n_01}, both: {n_11}, neither: {n_00}')
print(f'Net: {n_10 - n_01:+d}, chi2={chi2:.2f}, p={p_val:.4f} {sig}')

Failed SinglePass: 72
Rescued by Retry-NoMemory: 2
Rescued by Retry-CVX-Causal: 1
CVX-only rescues: 1
NoMem-only rescues: 2

=== McNemar: CVX-Causal vs NoMemory ===
CVX only: 3, NoMem only: 2, both: 26, neither: 69
Net: +1, chi2=0.00, p=1.0000 ns


In [9]:
import plotly.graph_objects as go

colors = {'SinglePass': '#95a5a6', 'Retry-NoMemory': '#3498db', 'Retry-CVX-Causal': '#2ecc71'}

fig = go.Figure()
for cond, res in all_results.items():
    rate = sum(r['passed'] for r in res) / len(res)
    fig.add_trace(go.Bar(
        x=[cond], y=[rate * 100],
        text=f'{rate:.1%}', textposition='outside',
        marker_color=colors.get(cond, '#333'),
    ))

fig.update_layout(
    title=f'APPS Introductory pass@1 (n={len(eval_problems)}, max {MAX_RETRIES} retries)',
    yaxis_title='pass@1 (%)', yaxis=dict(range=[0, 100]),
    height=400, showlegend=False,
)
fig.show()

In [10]:
print('=== E4: ITERATIVE CODING WITH CVX DEBUG MEMORY (APPS) ===')
print(f'Model: {LLM_MODEL}')
print(f'Debug corpus: {len(debug_traces)} multi-step traces from APPS introductory')
print(f'CVX index: {len(index)} vectors (6 steps per trace)')
print(f'Eval: APPS introductory (n={len(eval_problems)}), max {MAX_RETRIES} retries')
print()
for cond, res in all_results.items():
    passed = sum(r['passed'] for r in res)
    total = len(res)
    print(f'  {cond:>20}: {passed}/{total} = {passed/total:.1%}')
print()
print('CVX features: search across error/code/fix embeddings, episode encoding,')
print('multi-step temporal continuation (error→fix), iterative agent loop')

=== E4: ITERATIVE CODING WITH CVX DEBUG MEMORY (APPS) ===
Model: qwen2.5-coder:7b-instruct
Debug corpus: 173 multi-step traces from APPS introductory
CVX index: 1036 vectors (6 steps per trace)
Eval: APPS introductory (n=100), max 5 retries

            SinglePass: 28/100 = 28.0%
        Retry-NoMemory: 28/100 = 28.0%
      Retry-CVX-Causal: 29/100 = 29.0%

CVX features: search across error/code/fix embeddings, episode encoding,
multi-step temporal continuation (error→fix), iterative agent loop
